# Mervin/Mervis -- Mistral 7B LoRA fine-tune (Google Colab)

LoRA fine-tunes **Mistral-7B-Instruct v0.3** on the Mervin/Mervis persona
dataset with [Unsloth](https://github.com/unslothai/unsloth), exports a 4-bit
**Q4_K_M GGUF** (~4.4 GB), and uploads it to Hugging Face -- where `serve.py`
auto-downloads it like the other models.

**No system prompt.** The Mervin/Mervis tags and behavior come *purely* from
fine-tuning: training data is bare `user -> assistant`, so the model produces the
format with or without any system prompt. (Verified on an A100: correct output
either way.)

Trains on Google Colab (the other arena models used AWS SageMaker). A GPU is
required -- a free T4 is enough for 7B with Unsloth 4-bit; A100/L4 are faster.

### Before you run
- Runtime -> Change runtime type -> **GPU**.
- Add a Colab **secret** `HF_TOKEN` (key icon, left sidebar) with a Hugging Face
  **write** token. The upload cell reads it via `userdata`; it is never written
  into the notebook.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Unsloth + a dependency set it supports; drop the runtime's broken torchao
# (incompatible with the bleeding-edge torch here; optional for bnb 4-bit).
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0"
!pip uninstall -q -y torchao
import transformers, trl, datasets
print("transformers", transformers.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)

In [ ]:
import os
# Eager guard: some Colab torch images ship a broken torch.compile that crashes
# Unsloth at import (ImportError: CUSTOM_KEY). Run eager to be safe.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",  # not gated
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    full_finetuning = False,
)
print("loaded mistral-7b-instruct-v0.3")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset -- no system prompt

Each example is just `user -> assistant` rendered with Mistral's native template
(`<s>[INST] {prompt}[/INST] {response}</s>`). The persona is carried entirely by
the assistant targets, so the fine-tune -- not a system prompt -- is what makes
the model speak as Mervin/Mervis.

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"
raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

# Rows are 1-, 2-, or 3-turn (prompt/response, +prompt2/response2, +prompt3/response3).
# Render the WHOLE conversation so the model keeps the Mervin/Mervis format on
# FOLLOW-UP turns, not just turn 1.
def to_text(r):
    msgs = [{"role": "user", "content": r["prompt"]},
            {"role": "assistant", "content": r["response"]}]
    for i in ("2", "3"):
        p, a = (r.get("prompt" + i) or "").strip(), (r.get("response" + i) or "").strip()
        if p and a:
            msgs += [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = Dataset.from_list([to_text(r) for r in rows])
n2 = sum(1 for r in rows if r.get("prompt2")); n3 = sum(1 for r in rows if r.get("prompt3"))
print(f"{len(rows)-n2} single, {n2-n3} two-turn, {n3} three-turn")
print(ds[0]["text"][:500])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 6,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 5,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Sanity check -- bare user message, no system prompt

In [ ]:
# Both-tags check -- runs BEFORE export.
#   single + 2nd-turn = HARD GATE (raises -> a "Run all" STOPS here if a tag is dropped)
#   3rd-turn          = DIAGNOSTIC ONLY (reported, never blocks; longer context is best-effort)
import re
FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

def ask_msgs(msgs):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return text_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def both_tags(t):
    return bool(re.search(r"<Mervin>.*?</Mervin>", t, re.S) and re.search(r"<Mervis>.*?</Mervis>", t, re.S))

def convo(turns):
    msgs, reply = [], ""
    for u in turns:
        msgs.append({"role": "user", "content": u})
        reply = ask_msgs(msgs)
        msgs.append({"role": "assistant", "content": reply})
    return reply

SINGLE = ["What is 2+2?", "Tell me about Mondays.", "Recommend a book.", "Explain gravity."]
PAIRS = [("What do you think about Mondays?", "And what about Fridays?"),
         ("Should I get a cat?", "What about a dog instead?"),
         ("Is it going to rain tomorrow?", "And the day after?"),
         ("Tell me about the ocean.", "What lives in it?"),
         ("What is gravity?", "Does it ever take a day off?"),
         ("Recommend a movie.", "What about a cheerful one?")]
TRIPLES = [("What is 2+2?", "And times ten?", "Now minus three?"),
           ("Tell me about dogs.", "What about cats?", "And goldfish?"),
           ("Is coffee any good?", "What about tea?", "And plain water?"),
           ("Name a planet.", "Another one?", "The smallest?")]

t1 = t2 = t3 = 0
for q in SINGLE:
    r = convo([q]); g = both_tags(r); t1 += g
    print(("OK " if g else "BAD"), "T1 |", q, "->", r[:62])
for q1, q2 in PAIRS:
    r = convo([q1, q2]); g = both_tags(r); t2 += g
    print(("OK " if g else "BAD"), "T2 |", q2, "->", r[:62])
for q1, q2, q3 in TRIPLES:
    r = convo([q1, q2, q3]); g = both_tags(r); t3 += g
    print(("OK " if g else "BAD"), "T3 |", q3, "->", r[:62])

print(f"\nsingle {t1}/{len(SINGLE)}   2nd-turn {t2}/{len(PAIRS)} [GATE]   3rd-turn {t3}/{len(TRIPLES)} [diagnostic]")
assert t1 == len(SINGLE), f"single-turn regression: {t1}/{len(SINGLE)}"
assert t2 >= len(PAIRS) - 1, f"2ND-TURN REGRESSION: only {t2}/{len(PAIRS)} follow-up replies kept both tags. Do NOT upload."
print(f"GATE PASSED -- safe to export/upload. (3rd-turn {t3}/{len(TRIPLES)} informational only.)")


## Export Q4_K_M GGUF

Mistral converts cleanly to a real Q4_K_M GGUF (~4.4 GB). We quantize **before**
upload and push only the quantized file.

In [ ]:
import glob, os, time
t0 = time.time()
model.save_pretrained_gguf("mistral-merv-gguf", tokenizer, quantization_method="q4_k_m")
print("export done in", round(time.time()-t0), "s")
for f in sorted(glob.glob("/content/**/*Q4_K_M.gguf", recursive=True)):
    print(round(os.path.getsize(f)/1e9, 2), "GB ", f)

## Upload directly to Hugging Face

Pushes the quantized GGUF to `freeideas/merv-mistral7b` as `model-q4_k_m.gguf`,
where `serve.py` auto-downloads it. The token comes from the Colab `HF_TOKEN`
secret -- never written into the notebook.

In [ ]:
import glob
from google.colab import userdata
from huggingface_hub import HfApi

tok  = userdata.get("HF_TOKEN")          # write token, set in Colab Secrets
repo = "freeideas/merv-mistral7b"
src  = glob.glob("/content/**/mistral*Q4_K_M.gguf", recursive=True)[0]

api = HfApi()
print("uploading as model-q4_k_m.gguf ->", repo, "( as", api.whoami(token=tok)["name"], ")")
api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-q4_k_m.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` carry a `mistral` entry, so any host running
`serve.py` auto-downloads `model-q4_k_m.gguf` from `freeideas/merv-mistral7b` on
startup and the model appears in the dropdown. See this folder's `README.md`.